# Derived variables

This notebook shows how to use derived variables. A derived variable is a variable that is not available as an input dataset, but computed from one or more input variables.

In [1]:
import yaml

import esmvalcore.preprocessor
from esmvalcore.cmor.table import get_tables
from esmvalcore.config import CFG
from esmvalcore.dataset import Dataset, datasets_to_recipe

First, we configure ESMValCore so it searches the ESGF for data:

In [2]:
CFG["projects"]["CMIP6"].pop(
    "data",
    None,
)  # Clear existing CMIP6 configuration for finding input data
CFG.nested_update(
    {
        "projects": {
            "CMIP6": {
                "data": {
                    "intake-esgf": {
                        "type": "esmvalcore.io.intake_esgf.IntakeESGFDataSource",
                        "priority": 2,
                        "facets": {
                            "activity": "activity_drs",
                            "dataset": "source_id",
                            "ensemble": "member_id",
                            "exp": "experiment_id",
                            "institute": "institution_id",
                            "grid": "grid_label",
                            "mip": "table_id",
                            "project": "project",
                            "short_name": "variable_id",
                        },
                    },
                },
            },
        },
    },
)

## Which variables can be derived?

The interface for working with derived variables from Python is not very polished yet. To list all available derived variables, we can run:

In [3]:
list(esmvalcore.preprocessor._derive.ALL_DERIVED_VARIABLES)  # noqa: SLF001

['clmmtisccp',
 'sfcwind',
 'clhtkisccp',
 'rsntcs',
 'swcre',
 'hfns',
 'troz',
 'cllmtisccp',
 'lwcre',
 'hurs',
 'rsus',
 'clltkisccp',
 'rlntcs',
 'rlnst',
 'sispeed',
 'sm',
 'soz',
 'rtnt',
 'xco2',
 'vegfrac',
 'lvp',
 'toz',
 'rsnstcsnorm',
 'ohc',
 'xch4',
 'rsnst',
 'lapserate',
 'rsnt',
 'rsns',
 'alb',
 'siextent',
 'netcre',
 'chlora',
 'rlnstcs',
 'rsnstcs',
 'uajet',
 'clmtkisccp',
 'sithick',
 'amoc',
 'rlns',
 'co2s',
 'lwp',
 'ctotal',
 'qep',
 'asr',
 'rlus',
 'clhmtisccp',
 'et']

Note that [modules, functions, and variables starting with a single `_` character should be considered internal](https://peps.python.org/pep-0008/#descriptive-naming-styles) and there are no guarantees about the stability of this interface. Guidance on adding new derived variables to ESMValCore is available in [Deriving a variable](https://docs.esmvaltool.org/projects/ESMValCore/en/latest/develop/derivation.html).

## Finding available datasets

We define a dataset template to search for all CMIP6 models that provide all required input datasets to derive `lwcre` or longwave cloud radiative effect at the top of atmosphere on a monthly resolution for the historical experiment. Note that ESMValCore uses its own names for the facets for a more uniform naming across different CMIP phases and other projects. The mapping to the facet names used on ESGF can be found in [Facets](https://docs.esmvaltool.org/projects/ESMValCore/en/latest/reference/facets.html).

In [4]:
dataset_template = Dataset(
    short_name="lwcre",
    mip="Amon",
    project="CMIP6",
    exp="historical",
    dataset="*",
    institute="*",
    ensemble="r1i1p1f1",
    grid="gn",
)

Next, we use the `Dataset.derived_variable_from_files` method to build a list of datasets from the available files. This may take a while as searching the ESGF for many files may be a bit slow. Because the search results are cached, subsequent searches will be faster.

In [5]:
datasets = list(dataset_template.derived_variable_from_files())
print(f"Found {len(datasets)} datasets, showing the first 3 pairs:")
datasets[:3]

Found 37 datasets, showing the first 3 pairs:


[(Dataset:
  {'dataset': 'TaiESM1',
   'project': 'CMIP6',
   'mip': 'Amon',
   'short_name': 'rlut',
   'ensemble': 'r1i1p1f1',
   'exp': 'historical',
   'grid': 'gn',
   'institute': 'AS-RCEC'},
  Dataset:
  {'dataset': 'TaiESM1',
   'project': 'CMIP6',
   'mip': 'Amon',
   'short_name': 'rlutcs',
   'ensemble': 'r1i1p1f1',
   'exp': 'historical',
   'grid': 'gn',
   'institute': 'AS-RCEC'}),
 (Dataset:
  {'dataset': 'AWI-CM-1-1-MR',
   'project': 'CMIP6',
   'mip': 'Amon',
   'short_name': 'rlut',
   'ensemble': 'r1i1p1f1',
   'exp': 'historical',
   'grid': 'gn',
   'institute': 'AWI'},
  Dataset:
  {'dataset': 'AWI-CM-1-1-MR',
   'project': 'CMIP6',
   'mip': 'Amon',
   'short_name': 'rlutcs',
   'ensemble': 'r1i1p1f1',
   'exp': 'historical',
   'grid': 'gn',
   'institute': 'AWI'}),
 (Dataset:
  {'dataset': 'AWI-ESM-1-1-LR',
   'project': 'CMIP6',
   'mip': 'Amon',
   'short_name': 'rlut',
   'ensemble': 'r1i1p1f1',
   'exp': 'historical',
   'grid': 'gn',
   'institute': 'AWI'

This returned many tuples with input datasets required to derive `lwcre`. We can see `lwcre` is derived from the variables `rlut` and `rlutcs`.

## Composing a recipe with derived variables

To use the datasets found above in a recipe, we will want to use the name of the variable that needs to be derived, along with the `derive: true` option:

In [6]:
recipe_datasets = [
    input_datasets[0].copy(
        short_name="lwcre",
        diagnostic="diagnostic_name",
        derive=True,
    )
    for input_datasets in datasets
]
print(yaml.safe_dump(datasets_to_recipe(recipe_datasets)))

datasets:
- dataset: ACCESS-CM2
  institute: CSIRO-ARCCSS
- dataset: ACCESS-ESM1-5
  institute: CSIRO
- dataset: AWI-CM-1-1-MR
  institute: AWI
- dataset: AWI-ESM-1-1-LR
  institute: AWI
- dataset: BCC-CSM2-MR
  institute: BCC
- dataset: BCC-ESM1
  institute: BCC
- dataset: CAMS-CSM1-0
  institute: CAMS
- dataset: CAS-ESM2-0
  institute: CAS
- dataset: CESM2
  institute: NCAR
- dataset: CESM2-FV2
  institute: NCAR
- dataset: CESM2-WACCM
  institute: NCAR
- dataset: CESM2-WACCM-FV2
  institute: NCAR
- dataset: CMCC-CM2-HR4
  institute: CMCC
- dataset: CMCC-CM2-SR5
  institute: CMCC
- dataset: CMCC-ESM2
  institute: CMCC
- dataset: CanESM5
  institute: CCCma
- dataset: CanESM5-1
  institute: CCCma
- dataset: FGOALS-g3
  institute: CAS
- dataset: FIO-ESM-2-0
  institute: FIO-QLNM
- dataset: GISS-E2-1-G
  institute: NASA-GISS
- dataset: GISS-E2-1-G-CC
  institute: NASA-GISS
- dataset: GISS-E2-1-H
  institute: NASA-GISS
- dataset: GISS-E2-2-G
  institute: NASA-GISS
- dataset: GISS-E2-2-H
  

There is also a `force_derivation` option available for use in the recipe, when set to `true` that will cause the variable to be derived even if it is already available as a dataset.

## Computing the derived variable

Let's load the data to derive the first dataset:

In [7]:
cubes = [d.load() for d in datasets[0]]
cubes

 rlut: attribute positive not present
loaded from file 
 rlutcs: attribute positive not present
loaded from file 


[<iris 'Cube' of toa_outgoing_longwave_flux / (W m-2) (time: 1980; latitude: 192; longitude: 288)>,
 <iris 'Cube' of toa_outgoing_longwave_flux_assuming_clear_sky / (W m-2) (time: 1980; latitude: 192; longitude: 288)>]

Because the interface for using derived variables from Python isn't very polished yet, we need to pass some arguments in that can be retrieved from the CMOR table:

In [8]:
var_info = get_tables(CFG, project="CMIP6").get_variable(
    table_name="Amon",
    short_name="lwcre",
    derived=True,
)
kwargs = {
    k: getattr(var_info, k) for k in ["short_name", "long_name", "units"]
}
kwargs

{'short_name': 'lwcre',
 'long_name': 'TOA Longwave Cloud Radiative Effect',
 'units': 'W m-2'}

Now we are ready to derive the variable:

In [9]:
cube = esmvalcore.preprocessor.derive(cubes, **kwargs)
cube

<iris 'Cube' of TOA Longwave Cloud Radiative Effect / (W m-2) (time: 1980; latitude: 192; longitude: 288)>